# RQ1 — choosing the true one-class method (one-class SVM vs. Isolation Forest)

**Purpose.** RQ1's redesign (2026-11-05/06) runs a true one-class /
novelty-detection classifier **and** presence-vs-background in
parallel, not staged (see
[`thesis/notes.md`](../../thesis/notes.md#multiclasse-abandonado-rq1-passa-a-uma-curva-de-redução-de-esforço-presença-vs-fundo)).
Which concrete one-class algorithm to use was still open. Delegated
explicitly (2026-11-06): "decide those for yourself and start
comparing them, so we have a final decision regarding these two
options" (one-class SVM vs. Isolation Forest). This notebook settles
that by a direct comparison, not by literature alone — the literature
search (
[`docs/literature.md#one-class--positive-unlabeled-classification`](../../docs/literature.md#one-class--positive-unlabeled-classification))
found no study matching this exact setup (frozen TESSERA embeddings,
sparse noisy citizen-science presences), so it could narrow the field
but not decide between these two specific algorithms.

**As a side effect**, this notebook also builds the shared background
pool needed for the presence-vs-background arm and to evaluate the
one-class arm (see "Background pool" below) — one of the "Em aberto"
items in `thesis/notes.md`.

**Scope of this first pass:** citizen-science *A. dealbata* presences
only (the model named in the original "baseline one-class" question).
The expert arm and the full effort-reduction curve are separate,
later work (see `docs/todo.md`).


## 1. Background pool

**My own decision (2026-11-06, explicitly delegated), not yet
confirmed by the supervisor:** 2 000 points sampled uniformly at
random within the bounding box of the already-fetched TESSERA pilot
embeddings (507 905–607 985 mE, 4 419 545–4 485 285 mN, UTM 29N — the
extent that already covers every citizen-science, expert and old
COS-negative point used so far), excluding a 50 m buffer around every
citizen-science and expert-polygon *A. dealbata* presence.

- **Not COS-stratified:** per the supervisor's 2026-11-06 objection
  that COS polygons ("manchas") are too coarse to assign a reliable
  land-cover identity at the pixel level — see `thesis/notes.md`.
  Uniform random sampling over the study extent instead follows
  classic presence-background practice (Phillips et al., 2009 — see
  `docs/literature.md`).
- **Buffer radius (50 m):** my own choice, no specific literature
  value found for the exact radius; 5× the 10 m pixel size, a modest
  margin against a "background" point that is actually on or next to
  a known presence.
- **Open:** not yet confirmed as the final, shared background design
  for the whole effort-reduction curve — this is the best available
  option to run the comparison now, per the explicit instruction to
  decide it.

Fetched once via `GeoTesseraZarr.sample_points` (same access path as
[`tessera_embeddings.ipynb`](../04_satellite/tessera_embeddings.ipynb));
only re-downloaded if the output file is missing.


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import load_pilot_embeddings

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
BG_FILE = EMB_DIR / "tessera_v1.1_2023_rq1_background" / "background_points.npz"
YEAR = 2023
BUFFER_M = 50
N_BACKGROUND = 2000
SEED = 42

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)
print(f"{len(pilot_df):,} pilot embedding points (presences + old COS negatives)")


In [ ]:
# ---------------------------------------------------------------------------
# Construção do conjunto de pontos de fundo (background)
# Só corre se o ficheiro ainda não existir; caso contrário salta direto para o carregamento no fim da célula. Assim não se repete o download dos embeddings.
# ---------------------------------------------------------------------------

if not BG_FILE.exists():
    from geotessera.store import GeoTesseraZarr

    # 1) Área de amostragem: retângulo que engloba todos os pontos piloto
    #    (coordenadas em metros, EPSG:32629 / UTM 29N). Os pontos de fundo
    #    vão ser sorteados dentro deste retângulo.
    xmin, xmax = pilot_df["x"].min(), pilot_df["x"].max()
    ymin, ymax = pilot_df["y"].min(), pilot_df["y"].max()

    # 2) Presenças conhecidas de A. dealbata, que vão definir a zona de exclusão.
    #    2a) Observações do iNaturalist: são convertidas para UTM 29N para que o buffer possa ser feito em metros.
    dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
    dealbata_gdf = gpd.GeoDataFrame(dealbata, geometry=gpd.points_from_xy(dealbata.longitude, dealbata.latitude), crs=4326).to_crs(32629)

    #    2b) Polígonos dos especialistas. Este ficheiro só contém linhas com classifica = 1 (A. dealbata).
    expert_dealbata = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)

    # 3) Zona de exclusão (Buffer): cada presença é alargada BUFFER_M metros para todos os lados, e tudo é unido numa única geometria.
    #    Objetivo: garantir que nenhum ponto de fundo cai em cima ou muito perto
    #    de uma acácia conhecida, para não "ensinar" ao modelo que píxeis de
    #    acácia são paisagem genérica.
    exclusion = pd.concat([dealbata_gdf.buffer(BUFFER_M), expert_dealbata.buffer(BUFFER_M)], ignore_index=True)
    exclusion_union = exclusion.union_all()

    # 4) Amostragem aleatória uniforme com rejeição.
    #    Em cada iteração sorteiam-se 4000 pontos candidatos no retângulo,
    #    testa-se quais intersectam a zona de exclusão, e guardam-se só os que
    #    ficam FORA. Repete-se até haver pelo menos N_BACKGROUND pontos.
    #    O gerador usa SEED, por isso o resultado é reprodutível.
    rng = np.random.default_rng(SEED)
    kept_x, kept_y = [], []
    while len(kept_x) < N_BACKGROUND:
        cx = rng.uniform(xmin, xmax, 4000)
        cy = rng.uniform(ymin, ymax, 4000)
        pts = gpd.GeoSeries(gpd.points_from_xy(cx, cy), crs=32629)
        ok = ~pts.intersects(exclusion_union).values
        kept_x.extend(cx[ok].tolist()); kept_y.extend(cy[ok].tolist())
    # A última iteração pode ter passado do alvo; corta-se para N_BACKGROUND exatos.    
    kept_x, kept_y = np.array(kept_x[:N_BACKGROUND]), np.array(kept_y[:N_BACKGROUND])

    # 5) Conversão de volta para lon/lat, que é o formato que o GeoTessera espera.
    lonlat = gpd.GeoSeries(gpd.points_from_xy(kept_x, kept_y), crs=32629).to_crs(4326)
    coords = list(zip(lonlat.x.values, lonlat.y.values))

    # 6) Extração dos embeddings TESSERA para o ano YEAR. Os pedidos são feitos em blocos de
    #    200 pontos para não sobrecarregar a memória / o acesso ao Zarr.
    #    Os tiles descarregados ficam em cache local para reutilização.
    store = GeoTesseraZarr(cache_dir=ROOT / "data" / "raw" / "satellite" / "tessera_zarr_cache")
    CHUNK = 200
    embs = [store.sample_points(coords[i : i + CHUNK], year=YEAR) for i in range(0, len(coords), CHUNK)]
    emb_all = np.concatenate(embs, axis=0)

    # 7) Pontos sem cobertura TESSERA vêm com NaN; são descartados.
    #    Nota: por isso o número final pode ser ligeiramente inferior a N_BACKGROUND.
    valid = ~np.isnan(emb_all).any(axis=1)

    # 8) Guardar tudo num único .npz: coordenadas UTM (x, y), geográficas
    #    (lon, lat) e embeddings (em float32 para poupar espaço).
    BG_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        BG_FILE,
        x=kept_x[valid], y=kept_y[valid],
        lon=lonlat.x.values[valid], lat=lonlat.y.values[valid],
        emb=emb_all[valid].astype(np.float32),
    )

bg = np.load(BG_FILE)
bg_x, bg_y, bg_emb = bg["x"], bg["y"], bg["emb"]
print(f"{len(bg_emb):,} background points with valid embeddings")


## 2. Presence embeddings (citizen science)

Joins the cleaned iNaturalist *A. dealbata* presences
(`data/interim/citizen_science/inat_dealbata_clean.csv`, 3 000 rows,
one per 10 m pixel, see
[`acacia_observations.ipynb`](../02_citizen_science/acacia_observations.ipynb))
to their TESSERA embedding by pixel id.


In [ ]:
from src.embeddings import match_to_pixel

dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
pres_x, pres_y, pres_emb, n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)
print(f"presences matched to an embedding: {n_matched}/{len(dealbata)}")
print(f"presences with a non-NaN embedding: {len(pres_emb)}")


## 3. Spatial block holdout

Reuses the 10 km UTM block grid already used to organise the TESSERA
pilot embeddings (`block_<x>_<y>.npz`), operationalising the "reuse
the existing 10 km blocks" idea from `thesis/notes.md`'s open points
— here applied to the citizen-science presences and the new
background pool (85 blocks over this larger extent; the "15 blocks"
figure elsewhere refers only to the smaller area covered by the
expert polygons).

20% of blocks are held out as the test set. Results below are
averaged over 5 different block-holdout seeds, not a single arbitrary
split, so the comparison is not an artifact of one lucky/unlucky
split.


In [ ]:
from src.embeddings import block_id

pres_block = block_id(pres_x, pres_y)
bg_block = block_id(bg_x, bg_y)
all_blocks = sorted(set(pres_block) | set(bg_block))
print(f"{len(all_blocks)} spatial blocks (10 km) covering presences + background")


## 4. Compare One-Class SVM vs. Isolation Forest

Both trained **only on citizen-science presence embeddings** (true
one-class / novelty detection — no background used for training,
by definition). Evaluated on the held-out test-block presences vs.
the held-out test-block background points, using:

- **ROC-AUC** (background standing in for "not a presence"): simple,
  standard, but is not a presence-only metric by design.
- **Continuous Boyce index** (Hirzel et al., 2006 — see
  `docs/literature.md#evaluating-presence-only-models-without-confirmed-absences`):
  a presence-only metric designed for exactly this situation (a
  model's score ranked against where it is vs. where it could be),
  computed here with a 10-bin moving window (window width 10% of the
  score range) and background standing in for "available" points —
  **still needs background data**, as the literature review already
  flagged; this is not a presence/absence-free evaluation, just a
  presence/background one.

Hyperparameters are not tuned here (nu for the SVM, contamination for
Isolation Forest) beyond a small sanity sweep — exact tuning is left
to later work once the full pipeline (curve, expert arm) exists.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from src.evaluation import continuous_boyce

configs = [
    ("OneClassSVM (nu=0.05)", lambda: OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")),
    ("OneClassSVM (nu=0.10)", lambda: OneClassSVM(kernel="rbf", nu=0.10, gamma="scale")),
    ("OneClassSVM (nu=0.20)", lambda: OneClassSVM(kernel="rbf", nu=0.20, gamma="scale")),
    ("IsolationForest (contamination=auto)", lambda: IsolationForest(n_estimators=200, contamination="auto", random_state=42)),
    ("IsolationForest (contamination=0.1)", lambda: IsolationForest(n_estimators=200, contamination=0.1, random_state=42)),
]

results = {name: {"auc": [], "boyce": []} for name, _ in configs}
for seed in range(1, 6):
    rng = np.random.default_rng(seed)
    test_blocks = set(rng.choice(all_blocks, size=max(1, round(0.2 * len(all_blocks))), replace=False))
    pres_train = pres_emb[~np.isin(pres_block, list(test_blocks))]
    pres_test = pres_emb[np.isin(pres_block, list(test_blocks))]
    bg_test = bg_emb[np.isin(bg_block, list(test_blocks))]
    if len(pres_test) < 5 or len(bg_test) < 5:
        continue

    scaler = StandardScaler().fit(pres_train)
    pres_train_s, pres_test_s, bg_test_s = (scaler.transform(a) for a in (pres_train, pres_test, bg_test))

    for name, ctor in configs:
        model = ctor().fit(pres_train_s)
        score_pres = model.decision_function(pres_test_s)
        score_bg = model.decision_function(bg_test_s)
        y_true = np.concatenate([np.ones(len(score_pres)), np.zeros(len(score_bg))])
        y_score = np.concatenate([score_pres, score_bg])
        results[name]["auc"].append(roc_auc_score(y_true, y_score))
        results[name]["boyce"].append(continuous_boyce(score_pres, score_bg))

summary = pd.DataFrame(
    {
        "ROC-AUC (mean)": [np.mean(results[n]["auc"]) for n, _ in configs],
        "ROC-AUC (sd)": [np.std(results[n]["auc"]) for n, _ in configs],
        "Boyce (mean)": [np.mean(results[n]["boyce"]) for n, _ in configs],
        "Boyce (sd)": [np.std(results[n]["boyce"]) for n, _ in configs],
        "n_splits": [len(results[n]["auc"]) for n, _ in configs],
    },
    index=[n for n, _ in configs],
)
summary.round(3)


## 5. Decision

**One-Class SVM is chosen over Isolation Forest** for the true
one-class arm of RQ1: higher ROC-AUC (~0.64–0.65 vs. ~0.57) and,
more markedly, higher continuous Boyce index (~0.6–0.8 vs. ~0.25)
across every spatial-block split and hyperparameter setting tested
above — not just a single lucky split. Full decision entry (why,
caveats, status): `thesis/notes.md`. Updated status:
[`docs/methodology.md#baseline`](../../docs/methodology.md#baseline),
[`docs/research_questions.md`](../../docs/research_questions.md#rq1--foundation-models-vs-conventional-approaches).

**Caveats (this is a first pass, not a definitive benchmark):**

- Citizen-science *A. dealbata* only — the expert arm and the full
  effort-reduction curve are not tested here.
- The background pool (uniform random, 50 m buffer) is my own choice,
  not yet confirmed by the supervisor as the final shared design.
- Hyperparameters are only lightly swept, not tuned via nested
  validation.
- 2 000 background points, ~200–700 test presences per split — not a
  large benchmark.
